In [ ]:
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import prism
import warnings

from imagematerials.changedata import change_sector, ChangeAction, ChangeReplace
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import GenericMaterials, GenericStocks
from imagematerials.preprocessing import get_preprocessing_data


warnings.filterwarnings("ignore")

In [ ]:
# Get the preprocessing data for the vehicles sector only once
vhc_sector = get_preprocessing_data(
    "vehicles", Path("..", "data", "raw"), 
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"), 
    circular_economy_scenario_dirs = None
)

In [ ]:
vhc_sector.prep_data["material_fractions"]

In [ ]:
from monte_carlo import load_ranges, sample_intensities
import numpy as np

ranges = load_ranges("test_all_vehicles_material_ranges.csv")
rng = np.random.default_rng(42)   # seed once for reproducibility

all_output = {}
for i in range(3):
    mi = sample_intensities(ranges, rng=rng)   # same shape as original
    change_definition = {
        "material_fractions": ChangeReplace(mi)
    }
    factory = ModelFactory(
        vhc_sector, complete_timeline
        ).add(GenericStocks
        ).add(GenericMaterials
        ).add(Maintenance
        )
    model = factory.finish()
    model.simulate(simulation_timeline)
    all_output[i] = model

In [ ]:
@dataclass
class ChangeFirstElementIn3DArray(ChangeAction):
    new_value: float

    def apply(self, value):
        value[0, 0, 0] = self.new_value
        return value

In [ ]:
list_of_values = [0.42, 0.41, 0.40, 0.39, 0.38]
for value in list_of_values:
    change_definition = {
        "material_fractions": ChangeFirstElementIn3DArray(value)
    }
    new_vhc_sector = change_sector(vhc_sector, change_definition, inplace=False)
    print(
        f"Old value: {float(vhc_sector.prep_data['material_fractions'][0, 0, 0])};"
        f" new value: {float(new_vhc_sector.prep_data['material_fractions'][0, 0, 0])}."
    )
    # ... and then run the model.